<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Retail demand with a hierarchy

**[Retail demand with a hierarchy](https://learning.nextia-ai.com/courses/forecasting/m09/retail-demand-with-a-hierarchy/)** · Forecasting and Time Series: Predict What Comes Next · Case study 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** forecast a real online shop's weekly sales by country, product group and product, clean its cancellations honestly, compare slow-seller methods in the course's backtest, reconcile the hierarchy, and set a stock order from a quantile with its cost.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about 2 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The backtest and WAPE, from [Test like the future](https://learning.nextia-ai.com/courses/forecasting/m03/test-like-the-future-backtesting/) and [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). Reconciliation, from [Hierarchies that must add up](https://learning.nextia-ai.com/courses/forecasting/m06/hierarchies-that-must-add-up/). Croston, SBA and TSB, from [Intermittent demand](https://learning.nextia-ai.com/courses/forecasting/m06/intermittent-demand/). |
| **Data** | UCI Online Retail II, CC BY 4.0: real, anonymous invoice lines of a UK online gift shop, December 2009 to December 2011 |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1 and hierarchicalforecast 1.5.3, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m09/retail-demand-with-a-hierarchy/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M09-L03-retail-demand-with-a-hierarchy/retail-demand-with-a-hierarchy-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, hierarchicalforecast 1.5.3, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 hierarchicalforecast==1.5.3 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import hierarchicalforecast; print("hierarchicalforecast", hierarchicalforecast.__version__)

## Setup: get the data

Run the next cell. It downloads two files into a new folder `online-retail-forecasting` and checks each file's checksum:

- `online_retail_ii.csv.gz` (15 MB): the invoice lines of a real online shop, from the UCI Machine Learning Repository;
- `backtest.py`: the course's backtest helper, the same file as in every lesson of the course.

It needs an internet connection and no account, and takes about 10 seconds. You should see `Ready: online_retail_ii.csv.gz (15.0 MB) and backtest.py.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `online-retail-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

**Source and licence.** "Online Retail II" by Daqing Chen, UCI Machine Learning Repository, <https://doi.org/10.24432/C5CG6D>, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Nextia Learning joined the two Excel sheets into one CSV file and changed nothing else ([dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C05/case-studies/data/online_retail_ii.md)).

In [ ]:
# Setup: download the data and the backtest helper into online-retail-forecasting/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "online_retail_ii.csv.gz": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C05/case-studies/data/online_retail_ii.csv.gz",
        "c5ab45f73700a1411440c2cd6b7d85611d22cdc72b3594c2d29d37261e977e4f"
    ],
    "backtest.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/backtest.py",
        "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a"
    ]
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("online-retail-forecasting").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    if path.exists() and sha256(path) == expected:
        continue
    if (HERE / name).exists():  # a copy next to the notebook
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder online-retail-forecasting and run again.")

os.chdir(PROJECT)
print(f"Ready: online_retail_ii.csv.gz ({Path('online_retail_ii.csv.gz').stat().st_size / 1e6:.1f} MB) and backtest.py.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The invoice lines

One row is one line of an invoice: one product on one order. The next cell loads all 1,067,371 lines and counts the problems that matter for demand.

In [ ]:
lines = pd.read_csv("online_retail_ii.csv.gz", dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])
cancel = lines["Invoice"].str.startswith("C")
print(lines.shape, "|", lines["InvoiceDate"].min(), "to", lines["InvoiceDate"].max())
print(f"exact repeated lines: {lines.duplicated().sum():,} | cancellation lines (C...): {cancel.sum():,} | "
      f"price 0 or below: {(lines['Price'] <= 0).sum():,}")
print("lines per weekday (Monday = 0):", lines["InvoiceDate"].dt.dayofweek.value_counts().sort_index().to_dict())

> **Check.** You should see `(1067371, 8)`, from 2009-12-01 to 2011-12-09, 34,335 repeated lines, 19,494 cancellation lines and 6,207 lines with a price of 0 or below. Saturday (5) has almost no lines: the shop does not work on Saturdays.

## 2. Clean the lines

The first three rules are the same as in the machine learning course's case study *Segment online-shop customers*. The fourth rule is new, and it is about demand: an order that the customer cancelled within a day, for the same product and the same quantity, was a mistake. It was never demand, so both lines go.

In [ ]:
NOT_PRODUCTS = ["POST", "DOT", "C2", "C3", "M", "m", "D", "S", "B", "BANK CHARGES", "ADJUST",
                "ADJUST2", "AMAZONFEE", "CRUK", "TEST001", "TEST002", "GIFT"]
steps = []

def keep(df, rows_to_keep, reason):
    """Keep some lines, and record how many were removed and why."""
    steps.append({"removed because": reason, "lines removed": int((~rows_to_keep).sum()),
                  "lines left": int(rows_to_keep.sum())})
    return df[rows_to_keep]

clean = keep(lines, ~lines.duplicated(), "exact repeat of another line")
product = ~clean["StockCode"].isin(NOT_PRODUCTS) & ~clean["StockCode"].str.startswith("gift_")
clean = keep(clean, product, "not a product (postage, fee, voucher)")
clean = keep(clean, clean["Price"] > 0, "price 0 or below")
print(pd.DataFrame(steps).to_string(index=False))

> **Check.** You should see three rows, and `1021272` lines left after the last rule.

Now the cancellations. The next cell finds each cancellation that undoes an order: the same customer, the same product, the same quantity, at most 24 hours after the order.

In [ ]:
def undone_orders(clean):
    """Pairs of an order line and a cancellation of the same product, the same quantity and the same
    customer, within 24 hours after it. Returns the index labels of the order lines and of the cancellations."""
    c = clean[clean["Invoice"].str.startswith("C") & clean["Customer ID"].notna()]
    s = clean[~clean["Invoice"].str.startswith("C") & clean["Customer ID"].notna()]
    m = c.reset_index().merge(s.reset_index(), on=["Customer ID", "StockCode"], suffixes=("_c", "_s"))
    gap = m["InvoiceDate_c"] - m["InvoiceDate_s"]
    m = m[(m["Quantity_s"] == -m["Quantity_c"]) & (gap >= pd.Timedelta(0)) & (gap <= pd.Timedelta("1D"))]
    m = m.assign(gap=gap).sort_values("gap")
    m = m.drop_duplicates("index_c").drop_duplicates("index_s")  # one order for one cancellation
    return m["index_s"].tolist(), m["index_c"].tolist()

order_ix, cancel_ix = undone_orders(clean)
print(f"undone orders: {len(order_ix):,} order lines")
print(clean.loc[order_ix].nlargest(2, "Quantity")[["StockCode", "Description", "Quantity", "InvoiceDate"]].to_string())

> **Check.** You should see `undone orders: 1,331 order lines`, and the two largest: 80,995 paper crafts on 9 December 2011 and 74,215 storage jars on 18 January 2011. Each was cancelled minutes later.

> **Your turn.** How many **units** are in the undone orders? Add up the `Quantity` of the lines `order_ix` in `clean`, and put the number in `undone_units` as an `int`. Then run the check cell.

In [ ]:
undone_units = ...  # the units in the undone orders

In [ ]:
expect('undone_units', undone_units, 214327)

The other cancellations are real returns: a customer bought, then sent some back later. The order was demand at the time it was made, so we keep it. A returned unit goes back on the shelf, so it is not a negative order. The next cell builds the order lines.

In [ ]:
orders = clean.drop(index=order_ix)
orders = orders[~orders["Invoice"].str.startswith("C")]
cancel_units = int(-clean.loc[clean["Invoice"].str.startswith("C"), "Quantity"].sum())
print(f"order lines: {len(orders):,}, {int(orders['Quantity'].sum()):,} units | "
      f"other returns, not subtracted: {cancel_units - int(clean.loc[order_ix, 'Quantity'].sum()):,} units")

> **Check.** You should see `order lines: 1,002,026, 10,973,734 units` and `253,413` units of other returns.

## 3. Build the hierarchy

The shop has no product categories, so we build eight **product groups** from words in the description. The first group whose words appear wins. We keep five countries and put the others together. We count the orders per **week** (Monday to Sunday), from the first full week to the last full week, and take the 200 products with the most units **before the first test week**, so that the choice does not look at the future.

In [ ]:
COUNTRIES = ["United Kingdom", "EIRE", "Germany", "France", "Netherlands"]
GROUPS = [  # the first group whose words appear in the description wins
    ("christmas", ["CHRISTMAS", "XMAS", "ADVENT", "SANTA", "REINDEER"]),
    ("bags", ["BAG", "SHOPPER"]),
    ("party-and-baking", ["CAKE CASE", "PAPER CUP", "PAPER PLATE", "NAPKIN", "DOILIES", "PARTY", "BUNTING",
                          "CAKESTAND", "COOKIE", "BAKING"]),
    ("lights-and-candles", ["T-LIGHT", "CANDLE", "LIGHT", "LANTERN", "HLDR"]),
    ("kitchen-and-table", ["MUG", "BOWL", "LUNCH BOX", "SNACK BOX", "TEA", "JAR", "PLATE", "CUP", "GLASS"]),
    ("paper-and-crafts", ["WRAP", "CARD", "TISSUE", "PEN", "PENCIL", "TAG", "MEMO", "CHALK", "CRAYON",
                          "STICKER", "NOTEBOOK", "PAINT", "CRAFT"]),
    ("home-decor", ["SIGN", "FRAME", "HEART", "DECORATION", "MIRROR", "TRINKET", "WICKER", "HOLDER", "CLOCK",
                    "CUSHION", "DOOR"]),
]

def product_group(description):
    d = str(description).upper()
    for group, words in GROUPS:
        if any(w in d for w in words):
            return group
    return "other-gifts"

FIRST_WEEK, LAST_WEEK = pd.Timestamp("2009-12-07"), pd.Timestamp("2011-11-28")  # Mondays: full weeks only
ORIGINS = bt.origins("2011-10-03", n=7, every="28D")  # the comparison; then 4 weeks for the final test
FINAL_ORIGIN = pd.Timestamp("2011-10-31")
H = 4  # one order covers 4 weeks

orders["week"] = orders["InvoiceDate"].dt.normalize() - pd.to_timedelta(orders["InvoiceDate"].dt.dayofweek, unit="D")
orders = orders[(orders["week"] >= FIRST_WEEK) & (orders["week"] <= LAST_WEEK)]
orders["country"] = orders["Country"].where(orders["Country"].isin(COUNTRIES), "other countries")
before = orders[orders["week"] <= ORIGINS[0]]
top = before.groupby("StockCode")["Quantity"].sum().nlargest(200).index
names = before[before["StockCode"].isin(top)].groupby("StockCode")["Description"].agg(lambda s: s.mode().iat[0])
PRICES = before[before["StockCode"].isin(top)].groupby("StockCode")["Price"].median().to_dict()
groups = names.map(product_group)
print("products per group:", groups.value_counts().to_dict())

> **Check.** You should see 8 groups. The largest is `paper-and-crafts` with 40 products; the smallest is `christmas` with 6.

Now one series per country and product, with a 0 for every week without an order, and every level above it. hierarchicalforecast's `aggregate()` builds the levels and the summing matrix `S`.

In [ ]:
from hierarchicalforecast.utils import aggregate

sel = orders[orders["StockCode"].isin(top)]
weekly = sel.groupby(["country", "StockCode", "week"], as_index=False)["Quantity"].sum()
# every week for every country-product pair that had an order before the first test week
pairs = weekly[weekly["week"] <= ORIGINS[0]][["country", "StockCode"]].drop_duplicates()
grid = pairs.merge(pd.DataFrame({"week": pd.date_range(FIRST_WEEK, LAST_WEEK, freq="7D")}), how="cross")
bottom = grid.merge(weekly, on=["country", "StockCode", "week"], how="left").fillna({"Quantity": 0})
bottom = bottom.assign(group=bottom["StockCode"].map(groups), total="total").rename(
    columns={"StockCode": "product", "week": "ds", "Quantity": "y"})
spec = [["total"], ["total", "country"], ["total", "country", "group"], ["total", "country", "group", "product"]]
Y, S, tags = aggregate(bottom[["total", "country", "group", "product", "ds", "y"]], spec)
print({k: len(v) for k, v in tags.items()}, "| rows:", len(Y))

> **Check.** You should see `{'total': 1, 'total/country': 6, 'total/country/group': 48, 'total/country/group/product': 1048}` and `rows: 114712` (1,103 series × 104 weeks).

## 4. How slow are the products?

A product in one country is a **slow seller** when most weeks have no order. Two numbers sort the series into four classes (Syntetos and Boylan):

- **ADI**, the average demand interval: weeks per week with an order. Above 1.32: orders are not regular.
- **CV²**, the squared coefficient of variation of the order sizes. Above 0.49: the sizes vary a lot.

Smooth (both low), erratic (sizes vary), intermittent (gaps), lumpy (gaps and sizes vary).

> **Predict.** Of the 1,048 country-product series, how many are smooth? Then run the cell.

In [ ]:
def classify(y):
    """Syntetos-Boylan classes from ADI (weeks per week with an order) and CV2 (of the sizes)."""
    nz = y[y > 0]
    if len(nz) < 2:
        return "lumpy" if len(nz) else "no sale"
    adi = len(y) / len(nz)
    cv2 = (nz.std(ddof=0) / nz.mean()) ** 2
    return {(False, False): "smooth", (False, True): "erratic", (True, False): "intermittent",
            (True, True): "lumpy"}[(adi > 1.32, cv2 > 0.49)]

bottom_ids = tags["total/country/group/product"]
train0 = Y[(Y["ds"] <= ORIGINS[0]) & Y["unique_id"].isin(bottom_ids)]
kinds = train0.groupby("unique_id")["y"].apply(lambda s: classify(s.to_numpy()))
zero_share = train0.groupby("unique_id")["y"].apply(lambda s: float((s == 0).mean()))
print("all:", kinds.value_counts().to_dict(), "| mean share of zero weeks:", round(zero_share.mean(), 3))
print("United Kingdom:", kinds[kinds.index.str.startswith("total/United Kingdom/")].value_counts().to_dict())

> **Check.** You should see 49 smooth, 106 erratic, 427 intermittent and 466 lumpy series, and a mean share of zero weeks of 0.719. In the United Kingdom, most series are erratic or smooth. In the other countries, almost all are intermittent or lumpy.

> **Your turn.** What share of the 1,048 series is intermittent **or** lumpy? Put it in `slow_share`, rounded to 3 decimals. Then run the check cell.

In [ ]:
slow_share = ...  # the share of series that are intermittent or lumpy

In [ ]:
expect('slow_share', slow_share, 0.852, tolerance=0.0005)

## 5. Backtest the bottom series

The course backtest, with weeks: 7 origins 4 weeks apart, from 18 April to 3 October 2011, a horizon of 4 weeks, refit at every origin. Six methods: always 0, the seasonal naive forecast (the same week last year), ETS without a season (statsforecast's ETS cannot take a 52-week season), Croston, SBA and TSB.

The stock order is per **product**: the shop has one warehouse for all countries. So we add the country forecasts up per product and score the order there, with the course's costs: a unit short costs 35% of its price, a unit left over 25%.

> **Predict.** Which method gets the best MASE? Which gets the lowest cost? Then run the cell (about 15 seconds).

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, CrostonClassic, CrostonSBA, SeasonalNaive, ZeroModel

def stats(make_model):
    """A statsforecast model as a forecaster for bt.backtest()."""
    def forecaster(train, h, future):
        model = make_model()
        f = StatsForecast(models=[model], freq="7D", n_jobs=1).forecast(df=train[["unique_id", "ds", "y"]], h=h)
        return pd.DataFrame({"unique_id": f["unique_id"], "ds": f["ds"], "yhat": f[str(model)].clip(lower=0)})
    return forecaster

METHODS = {"always 0": ZeroModel, "seasonal naive (52 weeks)": lambda: SeasonalNaive(season_length=52),
           "ETS (no season)": lambda: AutoETS(season_length=1), "Croston": CrostonClassic, "SBA": CrostonSBA,
           "TSB": lambda: TSB(alpha_d=0.1, alpha_p=0.1)}
bot = Y[Y["unique_id"].isin(bottom_ids)].reset_index(drop=True)
product_of = {u: u.split("/")[-1] for u in bottom_ids}

def to_products(fc, column="yhat"):
    """Add the country series up to one series per product: the stock order is per product."""
    f = fc.assign(unique_id=fc["unique_id"].map(product_of))
    return f.groupby(["unique_id", "ds", "origin", "step"], as_index=False)[["y", column]].sum()

prod_hist = bot.assign(unique_id=bot["unique_id"].map(product_of)).groupby(["unique_id", "ds"], as_index=False)["y"].sum()
cost = bt.stock_cost(PRICES)
rows, forecasts = [], {}
for name, make in METHODS.items():
    forecasts[name] = bt.backtest(bot, stats(make), h=H, origins=ORIGINS)
    row = bt.score(forecasts[name], bot, name, season=52)
    product_row = bt.score(to_products(forecasts[name]), prod_hist, name, season=52, cost=cost)
    row["product WAPE %"], row["cost"] = product_row["WAPE %"], product_row["cost $"]
    rows.append(row)
table = pd.DataFrame(rows)[["method", "MASE", "WAPE %", "bias %", "product WAPE %", "cost"]]
print(table.to_string(index=False))

> **Check.** You should see six rows. `always 0` has the best MASE (0.986) and the worst cost (70,921). TSB has the lowest cost (25,035), the lowest WAPE (81.3%) and a bias of +0.7%. Croston is biased high (+21.9%). The costs are in pounds per 4-week order for the 200 products (the helper's column is named `cost $`).

## 6. Reconcile the hierarchy

ETS forecasts every one of the 1,103 series at every origin, then bottom-up and MinT make them add up. We also add up TSB's bottom forecasts: a bottom-up forecast that is coherent by construction. Each method gets a WAPE per level.

> **Predict.** At the total, will TSB added up beat ETS's own forecast of the total? Then run the cell (about 30 seconds).

In [ ]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

def reconcile_at(origin):
    """Base ETS forecasts for all series up to `origin`, reconciled bottom-up and with MinT."""
    sf = StatsForecast(models=[AutoETS(season_length=1)], freq="7D", n_jobs=1)
    base = sf.forecast(df=Y[Y["ds"] <= origin], h=H, fitted=True)
    base["AutoETS"] = base["AutoETS"].clip(lower=0)
    hrec = HierarchicalReconciliation(reconcilers=[BottomUp(), MinTrace(method="mint_shrink", nonnegative=True)])
    return hrec.reconcile(Y_hat_df=base, Y_df=sf.forecast_fitted_values(), S_df=S, tags=tags)

def add_up(fc, column="yhat"):
    """Add bottom forecasts up to every level with the summing matrix S."""
    Smat = S.set_index("unique_id")[bottom_ids]
    out = []
    for origin, f in fc.groupby("origin"):
        p = f.pivot(index="unique_id", columns="ds", values=column).reindex(bottom_ids).fillna(0)
        a = pd.DataFrame(Smat.to_numpy() @ p.to_numpy(), index=Smat.index, columns=p.columns)
        a = a.stack().rename("TSB bottom-up").reset_index().rename(columns={"level_1": "ds"})
        out.append(a.assign(origin=origin))
    return pd.concat(out, ignore_index=True)

rec = pd.concat([reconcile_at(o).assign(origin=o) for o in ORIGINS], ignore_index=True)
rec = rec.merge(Y, on=["unique_id", "ds"]).merge(add_up(forecasts["TSB"]), on=["unique_id", "ds", "origin"])
mint = [c for c in rec.columns if "MinTrace" in c][0]
COLS = {"ETS base (not coherent)": "AutoETS", "ETS bottom-up": "AutoETS/BottomUp", "ETS MinT": mint,
        "TSB bottom-up": "TSB bottom-up"}
LEVELS = {"total": "total", "total/country": "country", "total/country/group": "group in a country",
          "total/country/group/product": "product in a country"}

def wape(df, col):
    return round(float(100 * (df["y"] - df[col]).abs().sum() / df["y"].sum()), 1)

by_level = pd.DataFrame({name: {LEVELS[lv]: wape(rec[rec["unique_id"].isin(ids)], col) for lv, ids in tags.items()}
                         for name, col in COLS.items()}).T
print("WAPE % by level, 7 origins:")
print(by_level.to_string())

> **Check.** You should see four rows. The ETS base row is 16.3, 29.7, 45.4 and 88.0. TSB bottom-up is the best at every level: 15.7, 27.4, 42.1 and 81.3. MinT improves ETS a little at the three lower levels (29.0, 44.5, 86.9) but not at the total (17.3).

## 7. The stock order from a quantile

The order should cover the 4 weeks with the right chance of running out. With the course's costs, the best quantile to order is the **critical ratio** 0.35 / (0.35 + 0.25) = 0.583. We need a range for each product's 4-week demand. Two ways:

- **normal**: the forecast total ± the spread of the last 26 weeks, as if the weeks were independent and the demand symmetric;
- **empirical**: the product's own past 4-week totals in the last 52 weeks, scaled so that their mean equals the forecast. It keeps the skew of lumpy demand: many small totals, a few big ones.

In [ ]:
LOST, LEFT = 0.35, 0.25
RATIO = LOST / (LOST + LEFT)
from scipy.stats import norm

def normal_order(point, history, origin, ratio=RATIO, weeks=26):
    """The `ratio` quantile of a normal range around the 4-week forecast total."""
    tail = history[(history["ds"] <= origin) & (history["ds"] > origin - pd.Timedelta(weeks=weeks))]
    sd = tail.groupby("unique_id")["y"].std() * np.sqrt(H)
    total = point.groupby("unique_id")["yhat"].sum()
    return (total + norm.ppf(ratio) * sd.reindex(total.index).fillna(0)).clip(lower=0)

def empirical_order(point, history, origin, ratio=RATIO, weeks=52):
    """The `ratio` quantile of the product's past 4-week totals, scaled to the forecast total."""
    tail = history[(history["ds"] <= origin) & (history["ds"] > origin - pd.Timedelta(weeks=weeks))]
    sums = tail.sort_values("ds").groupby("unique_id")["y"].apply(lambda s: s.rolling(H).sum().dropna().to_numpy())
    total = point.groupby("unique_id")["yhat"].sum()
    out = {}
    for u, t in total.items():
        v = sums.get(u, np.array([]))
        out[u] = t * np.quantile(v, ratio) / v.mean() if len(v) and v.mean() > 0 else t
    return pd.Series(out).clip(lower=0)

def order_cost(fc_products, history, quantile, ratio=RATIO):
    """The cost per 4-week order of ordering the `ratio` quantile, and the share of orders that run out."""
    costs, short = [], []
    for origin, f in fc_products.groupby("origin"):
        q = np.ceil(quantile(f, history, origin, ratio))
        demand = f.groupby("unique_id")["y"].sum()
        p = pd.Series(PRICES).reindex(q.index)
        costs.append(float((np.maximum(demand - q, 0) * LOST * p + np.maximum(q - demand, 0) * LEFT * p).sum()))
        short.append(float((demand > q).mean()))
    return round(float(np.mean(costs))), round(float(np.mean(short)), 3)

def coverage80(fc_products, history, quantile):
    """The share of 4-week product totals inside the 10%-90% range."""
    inside = []
    for origin, f in fc_products.groupby("origin"):
        lo, hi = quantile(f, history, origin, 0.1), quantile(f, history, origin, 0.9)
        d = f.groupby("unique_id")["y"].sum()
        inside.append(((d >= lo) & (d <= hi)).mean())
    return round(float(np.mean(inside)), 3)

tsb_products = to_products(forecasts["TSB"])
d = tsb_products.groupby(["unique_id", "origin"])[["y", "yhat"]].sum()
print(f"critical ratio {RATIO:.3f}")
print(f"TSB, the forecast total:   cost {bt.score(tsb_products, prod_hist, 'TSB', season=52, cost=cost)['cost $']}, "
      f"runs out {(d['y'] > np.ceil(d['yhat'])).mean():.3f}")
for label, q in [("normal", normal_order), ("empirical", empirical_order)]:
    c, s = order_cost(tsb_products, prod_hist, q)
    print(f"TSB, {label:9} 58.3% order: cost {c}, runs out {s}, 80% range holds {coverage80(tsb_products, prod_hist, q)}")

> **Check.** You should see three lines. The forecast total costs 25,035 and runs out on 38.6% of the orders. The normal order costs 25,780; its 80% range holds only 65.4% of the real totals: too narrow. The empirical order costs 25,968, runs out on 42.3% of the orders (the theory says 41.7%), and its 80% range holds 77.1%.

The empirical range is close to honest, but its order still costs more than the plain forecast total. The next cell moves the quantile to see where the cost is lowest.

In [ ]:
curve = pd.DataFrame([(r, *order_cost(tsb_products, prod_hist, empirical_order, r))
                      for r in [0.3, 0.4, 0.5, 0.583, 0.6, 0.7, 0.8]], columns=["quantile", "cost", "runs out"])
print(curve.to_string(index=False))

> **Check.** You should see the lowest cost near the critical ratio: 25,967 at 0.583 and 25,896 at 0.6. The theory finds the right level for this range. But every row costs more than the forecast total (25,035).

> **Your turn.** Compute the cost of the **empirical** order at the median (`ratio=0.5`) for the **ETS** forecasts (`forecasts["ETS (no season)"]`). Put the cost in `ets_median_cost`. Then run the check cell.

In [ ]:
ets_median_cost = ...  # the cost of ETS's empirical order at the 0.5 quantile

In [ ]:
expect('ets_median_cost', ets_median_cost, 27657)

## 8. The final test, once

The comparison chose TSB, added up per product, with the forecast total as the order. Now the 4 weeks after 31 October 2011, which no step has seen: November, the weeks before Christmas. We run every method once and change nothing after.

In [ ]:
final = {}
for name in ["always 0", "seasonal naive (52 weeks)", "ETS (no season)", "SBA", "TSB"]:
    f = bt.backtest(bot, stats(METHODS[name]), h=H, origins=[FINAL_ORIGIN])
    r = bt.score(to_products(f), prod_hist, name, season=52, cost=cost)
    final[name] = {"product WAPE %": r["WAPE %"], "bias %": r["bias %"], "cost": r["cost $"]}
print(pd.DataFrame(final).T.astype({"cost": int}).to_string())

> **Check.** You should see TSB with a bias of −21.0% and a cost of 41,594, and ETS with −11.3% and 37,163. In November, ETS is cheaper. The chosen method lost the final test.

The Christmas season starts in the test weeks, and the comparison weeks (April to October) had no such ramp. TSB moves slowly (it gives each new week a weight of only 0.1), so it stays low while orders grow. We report this result and change nothing now. The lesson page discusses what to do next.

## 9. The order for December, on new input

The last step runs the chosen pipeline on all the weeks to 28 November 2011 and writes the order for the next 4 weeks. Then it reads the newest lines, 5 to 9 December 2011, with the same cleaning.

In [ ]:
sf = StatsForecast(models=[TSB(alpha_d=0.1, alpha_p=0.1)], freq="7D", n_jobs=1)
dec = sf.forecast(df=bot[["unique_id", "ds", "y"]], h=H)
dec["product"] = dec["unique_id"].map(product_of)
order = np.ceil(dec.groupby("product")["TSB"].sum()).astype(int)
print(f"December order: {order.sum():,} units for {len(order)} products")
print(pd.DataFrame({"description": names.reindex(order.nlargest(3).index), "order": order.nlargest(3)}).to_string())

new = lines[lines["InvoiceDate"] >= "2011-12-05"]
new = new[~new.duplicated() & ~new["StockCode"].isin(NOT_PRODUCTS) & ~new["StockCode"].str.startswith("gift_") & (new["Price"] > 0)]
o_ix, c_ix = undone_orders(new)
print(f"new lines 5-9 December: {len(new):,}; undone orders: {int(new.loc[o_ix, 'Quantity'].sum()):,} units")

> **Check.** You should see `December order: 165,642 units for 200 products`, with the popcorn holder first (7,329). The new lines contain the mistaken order of 80,995 paper crafts, and the same rule removes it: `undone orders: 81,162 units`.

Last December, the week of 20 December sold only 30.7% of the first week of December, and the week of 27 December sold nothing: the shop closes over Christmas. TSB does not know the calendar. The lesson page shows the order with the calendar.

## 10. Failure case: Croston and the mug that stopped selling

Croston changes its forecast only in a week with an order. Look at one product.

In [ ]:
mug = "37410"
print(names[mug], "| price", PRICES[mug])
print("weeks with an order:", int((prod_hist[prod_hist["unique_id"] == mug]["y"] > 0).sum()),
      "| last:", prod_hist[(prod_hist["unique_id"] == mug) & (prod_hist["y"] > 0)]["ds"].max().date())
for name in ["Croston", "TSB"]:
    f = to_products(forecasts[name]).query("unique_id == @mug")
    print(f"{name:8} forecast per week at the first origin: {f['yhat'].iloc[0]:7.1f} | "
          f"at the last: {f['yhat'].iloc[-1]:7.1f} | cost per order: {bt.stock_cost(PRICES)(f):,.0f}")

> **Check.** You should see 12 weeks with an order, the last one on 2010-04-05. Croston forecasts 1129.4 mugs a week at every origin and costs 1,412 pounds per order; TSB goes from 2.1 to 0.2 and costs 1.

One order of 19,152 mugs from Denmark in February 2010 set Croston's average order size. The mug then stopped selling, and Croston never noticed. TSB lowers its forecast in every week without an order.

## Next

You have taken a real shop's invoice lines to a stock order with its cost. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m09/retail-demand-with-a-hierarchy/) has the charts, the recommendation, the limits, how to make this a portfolio piece, and the knowledge checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/forecasting/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m09/retail-demand-with-a-hierarchy/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and anonymous: UCI Online Retail II, CC BY 4.0.